### 规格化巴西动物数据


In [1]:
import pandas as pd
# 函数
def num_trans(series):
    # 传入一个series，返回一个数值化后的series
    series = series.astype(str)
    series = series.str.replace(',', '')
    series = pd.to_numeric(series, errors='coerce')
    return series

In [3]:
import dis
dis.dis(num_trans)

  3           0 RESUME                   0

  5           2 LOAD_FAST                0 (series)
              4 LOAD_METHOD              0 (astype)
             26 LOAD_GLOBAL              2 (str)
             38 PRECALL                  1
             42 CALL                     1
             52 STORE_FAST               0 (series)

  6          54 LOAD_FAST                0 (series)
             56 LOAD_ATTR                1 (str)
             66 LOAD_METHOD              2 (replace)
             88 LOAD_CONST               1 (',')
             90 LOAD_CONST               2 ('')
             92 PRECALL                  2
             96 CALL                     2
            106 STORE_FAST               0 (series)

  7         108 LOAD_GLOBAL              6 (pd)
            120 LOAD_METHOD              4 (to_numeric)
            142 LOAD_FAST                0 (series)
            144 LOAD_CONST               3 ('coerce')
            146 KW_NAMES                 4
            148 PRECA

In [ ]:
# 读文件
source_path = "巴西动物_standard/1975.csv"
target_path = "动物_ok/1975.csv"
lines = []
with open(source_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

# 不要前4行,与后面的一些错误行
lines = lines[4:5572]
lines[0] = lines[0][0:-1]+',,""\n'
with open(target_path, "w", encoding="utf-8-sig") as file:
    file.writelines(lines)

In [95]:
import pandas as pd
year_begin = 1974
year_end = 2022

fixed_columns = ['"Level"',
 '"Code."',
 '"Municipality"',
 '"Cattle"',
 '',
 '"Buffalo"',
 '',
 '"Horse"',
 '',
 '"Pork - total"',
 '',
 '"Swine - swine breeding stock"',
 '',
 '"Goat"',
 '',
 '"Sheep"',
 '',
 '"Poultry - total"',
 '',
 '"Poultry - chickens"',
 '',
 '"Quails"',
 '']

for y in range(year_begin,year_end+1):

    source_path = "巴西动物_standard/"+str(y)+".csv"
    target_path = "动物_ok/"+str(y)+".csv"

    # 读取文件并去除前4行和错误行
    with open(source_path, "r", encoding="utf-8") as file:
        lines = file.readlines()[4:5572]
    lines[0] = lines[0][0:-1]+',\n'

    # 假设第一行是列标题，接下来是数据行
    # column_names = lines[0].strip().split(',') # 第一行作为列标题
    data_rows = [line.strip().split(',') for line in lines[1:]] # 剩下的行作为数据

    # 使用列标题和数据行创建DataFrame
    df = pd.DataFrame(data_rows, columns=fixed_columns)

    # 创建一个只包含非空列名的列表
    non_empty_columns = [col for col in df.columns if col.strip() != '']

    # 使用新的列名列表来选择 DataFrame 的列
    df = df[non_empty_columns]
    df.columns = df.columns.str.replace('"', '')
    df = df.applymap(lambda x: x.replace('"', '') if isinstance(x, str) else x)

    # 将数据数值化
    df['Sheep'] = num_trans(df['Sheep'])
    df['Goat'] = num_trans(df['Goat'])
    df['Cattle'] = num_trans(df['Cattle'])
    df['Buffalo'] = num_trans(df['Buffalo'])

    # 一些列进行合并
    df['Sheep'] = df['Sheep'] + df['Goat']
    df = df.drop(columns=['Goat'])
    df['Cattle'] = df['Cattle'] + df['Buffalo']
    df = df.drop(columns=['Buffalo'])

    df.to_csv(target_path, index=False, encoding="utf-8-sig")

In [3]:
import pandas as pd
# 与FAO数据校对
# 美国、巴西,单个国家的校对
def single_proofread(data,data_fao,data_fao_item="item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"

    for animal in animal_species:

        # 首先转化列里面数值保证能进行四则运算
        data[animal] = data[animal].astype(str)
        data[animal] = data[animal].str.replace(',', '')  # remove commas
        data[animal] = pd.to_numeric(data[animal], errors='coerce')
        data_fao[value_name] = data_fao[value_name].astype(str)
        data_fao[value_name] = data_fao[value_name].str.replace(',', '')
        data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

         # 首先计算data里面每个县占比
        proportions = data[animal] / data[animal].sum()

        # 如果FAO数据缺失那么就不进行校对
        value = data_fao[data_fao[data_fao_item]==animal][value_name].values[0]
        # 检查value是否为数值类型
        if isinstance(value, (int, float)):
            # 如果是数值类型，检查是否大于等于0
            if value > 0:
                data[animal] = proportions*value
    return data


In [16]:
# 规范化fao数据，注意单位
# 输入所有年份的fao原始数据,输出分年份规范化好的fao数据
def fao_standard(fao_data,params,target_path):
    # params是字典，键是fao数据的动物种类名称，值是键对应的要转化的种类名称，所有值是列表则证明要将这两项相加
    # 确保数值列可以做四则运算
    value_name = "Value" if "Value" in fao_data.columns else "value"
    fao_data[value_name] = fao_data[value_name].astype(str)
    fao_data[value_name] = fao_data[value_name].str.replace(',', '')
    fao_data[value_name] = pd.to_numeric(fao_data[value_name], errors='coerce')

    # 单位转化
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Value'] *= 1000
    fao_data.loc[fao_data['Unit'] == '1000 An', 'Unit'] = 'An'

    for year,group in fao_data.groupby("Year"):
        # 一年一年来，搞完保存

        # fao_data数据只有Item列需要,用replace替换
        for item in params:
            if isinstance(item,str):
                group["Item"] = group['Item'].replace(item,params[item])
            elif isinstance(item,tuple):
                # 先相加，按照年份，然后形成新的一个值
                for i in range(1,len(item)):
                    group.loc[group["Item"]==item[0],value_name] += group[group['Item']==item[i]][value_name].values[0]
                    
                group["Item"] = group['Item'].replace(item[0],params[item]) 
            else:
                print("params输入格式错误,错误的键为{}".format(item))

        # 保存
        group.to_csv(target_path+str(year)+".csv")

In [35]:
import pandas as pd
# 巴西的先把原始数据列种类整理
year_begin = 1974
year_end = 2022
for y in range(year_begin,year_end+1):
    data = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/"+str(y)+".csv")

    # 把cattle拆成beef_cattle与milk_cattle，excl_cattle，数值与cattle保持一致
    # data['beef_cattle'] = data['Cattle']
    # data['milk_cattle'] = data['Cattle']
    # data['excl_cattle'] = data['Cattle']
    # data['Layers'] = data['Poultry - chickens']
    # data['Broilers'] = data['Poultry - chickens']
    # data.drop(columns=['Cattle','Swine - swine breeding stock','Poultry - total','Quails','Poultry - chickens'],inplace=True)
    data.rename(columns={'Sheep': 'Sheep_Goat'}, inplace=True)
    data.to_csv("D:/中科院数据下载/巴西/动物_ok/"+str(y)+".csv",index=False,encoding='utf-8-sig')

In [36]:
params = {
    "Raw milk of cattle":"milk_cattle",
    "Meat of cattle with the bone, fresh or chilled":"beef_cattle",
    # "excl cattle":
    "Meat of pig with the bone, fresh or chilled":"Pork - total",
    ('Sheep','Goats'):"Sheep_Goat",
    "Hen eggs in shell, fresh":"Layers",
    "Meat of chickens, fresh or chilled":"Broilers",
    "Horse meat, fresh or chilled":'Horse'
}


In [37]:
import pandas as pd
# 规范化fao数据
target_path = 'D:/中科院数据下载/巴西/动物_ok/fao/'
fao_data = pd.read_csv('D:/中科院数据下载/巴西/动物_ok/fao/FAOSTAT_data_en_1-23-2024.csv')
fao_standard(fao_data,params,target_path)

In [39]:
# 与FAO数据校对,注意单位
# 美国、巴西,单个国家的校对
def single_proofread(data,data_fao,Year,data_fao_item="Item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    data['标记'] = ""
    count_zero = 0
    count_error = 0
    r = data.iloc[:,0:3] # 存储比例
    r['year'] = Year

    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            r[animal] = ''
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            data_fao[value_name] = data_fao[value_name].astype(str)
            data_fao[value_name] = data_fao[value_name].str.replace(',', '')
            data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

            # 首先计算data里面每个县占比
            country_total = data[animal].sum()
            if country_total != 0:
                proportions = data[animal] / country_total
                r[animal] = proportions
            else:
                continue

            # 如果FAO数据缺失那么就不进行校对
            fao_value = data_fao[data_fao[data_fao_item]==animal][value_name].values[0]

            if abs(fao_value-data[animal].sum()) > 0.2*max(fao_value,data[animal].sum()):
                count_error += 1
                if country_total==0 and fao_value != 0:
                    count_zero += 1
                    
                # fao总量与国家总量差别太大的不要
                    data["标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(data[animal].sum())+"; "

                # continue 

            # 检查value是否为数值类型
            if isinstance(fao_value, (int, float)):
                # 如果是数值类型，检查是否大于等于0
                if fao_value > 0:
                    data[animal] = proportions*fao_value
    return data,r


In [40]:
import pandas as pd
r = pd.DataFrame()

# 进行校对
# 与FAO数据校对
year_begin = 1974
year_end = 2022
for y in range(year_begin,year_end+1):
    data = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/"+str(y)+".csv")
    data_fao = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/fao/"+str(y)+".csv")
    data_ok,r_tmp = single_proofread(data,data_fao,y)

    r = pd.concat([r,r_tmp],axis=0)
    # data_ok.to_csv("动物_fao_ok/"+str(y)+".csv",index=False,encoding="utf-8-sig")

In [41]:
r.to_csv("D:/中科院数据下载/巴西/动物_ok/fao/比例县.csv",index=False,encoding='utf-8-sig')

In [44]:
# 与FAO数据校对,注意单位
# 美国、巴西,单个国家的校对
def single_proofread_r(data,data_fao,r,Year,data_fao_item="Item"):
    # data是之前已经整理好的数据，现在要被校对
    # data_fao是fao上的国家总量
    data['标记'] = ""
    count_zero = 0
    count_error = 0


    # 首先筛选出动物种类
    animal_species = data_fao[data_fao_item].unique() 
    value_name = "Value" if "Value" in data_fao.columns else "value"

    for animal in animal_species:
        
        # 首先判断这个动物种类是否在data里面
        if animal in data.columns:
            # 首先转化列里面数值保证能进行四则运算
            r[animal] = r[animal].astype(str)
            r[animal] = r[animal].str.replace(',', '')  # remove commas
            r[animal] = pd.to_numeric(r[animal], errors='coerce').fillna(0)

            # 首先转化列里面数值保证能进行四则运算
            data[animal] = data[animal].astype(str)
            data[animal] = data[animal].str.replace(',', '')  # remove commas
            data[animal] = pd.to_numeric(data[animal], errors='coerce')
            data_fao[value_name] = data_fao[value_name].astype(str)
            data_fao[value_name] = data_fao[value_name].str.replace(',', '')
            data_fao[value_name] = pd.to_numeric(data_fao[value_name], errors='coerce')

            # 首先计算data里面每个县占比
            country_total = data[animal].sum()
            if country_total != 0:
                proportions = data[animal] / country_total
              
            else:
                 # 找出最近的年份，其中 'animal' 列的值的总和大于等于 0
                for offset in range(1, max(Year - 1980, 2021 - Year) + 1):
                    recent_year = Year - offset if Year - offset >= 1980 else Year + offset
                    if recent_year <= 2021 and r[ r['year']==recent_year][animal].sum() >= 0:
                        proportions = r[r['year']==recent_year][animal]
                        break

            # 如果FAO数据缺失那么就不进行校对
            fao_value = data_fao[data_fao[data_fao_item]==animal][value_name].values[0]

            if abs(fao_value-data[animal].sum()) > 0.2*max(fao_value,data[animal].sum()):
                count_error += 1
                if country_total==0 and fao_value != 0:
                    count_zero += 1
                else:
                # fao总量与国家总量差别太大的不要
                    data["标记"] += "种类："+animal+","+"FAO总量："+str(fao_value)+","+"国家总量："+str(data[animal].sum())+"; "

                # continue 
            print("年份：{}，种类为{}的fao总量为{}，对应国家数据总量{}".format(Year,animal,fao_value,country_total))

            # 检查value是否为数值类型
            if isinstance(fao_value, (int, float)):
                # 如果是数值类型，检查是否大于等于0
                if fao_value > 0:
                    data[animal] = proportions*fao_value
    print("缺失占比为{}".format(count_zero/count_error))
    
    return data


In [45]:
import pandas as pd
r = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/fao/比例县.csv")

# 进行校对
# 与FAO数据校对
year_begin = 1974
year_end = 2022
for y in range(year_begin,year_end+1):
    data = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/"+str(y)+".csv")
    data_fao = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/fao/"+str(y)+".csv")
    data_ok = single_proofread_r(data,data_fao,r,y)

    # data_ok.to_csv("D:/中科院数据下载/巴西/动物_fao_ok/"+str(y)+".csv",index=False,encoding="utf-8-sig")

C:\Users\typing\AppData\Local\Temp\ipykernel_23224\3770566669.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  r = pd.read_csv("D:/中科院数据下载/巴西/动物_ok/fao/比例县.csv")


年份：1974，种类为Horse的fao总量为506000，对应国家数据总量5216543.0
年份：1974，种类为beef_cattle的fao总量为10500000，对应国家数据总量44577853.0
年份：1974，种类为Broilers的fao总量为483636000，对应国家数据总量112980458.0
年份：1974，种类为Pork - total的fao总量为10791000，对应国家数据总量34191986.0
年份：1974，种类为milk_cattle的fao总量为10838540，对应国家数据总量44577853.0
年份：1974，种类为Sheep_Goat的fao总量为24571000，对应国家数据总量26025527.0
缺失占比为0.0
年份：1975，种类为Horse的fao总量为484076，对应国家数据总量5506968.0
年份：1975，种类为beef_cattle的fao总量为11500000，对应国家数据总量51705945.0
年份：1975，种类为Broilers的fao总量为485455000，对应国家数据总量133441993.0
年份：1975，种类为Pork - total的fao总量为11343000，对应国家数据总量37640291.0
年份：1975，种类为milk_cattle的fao总量为12293660，对应国家数据总量51705945.0
年份：1975，种类为Sheep_Goat的fao总量为24929224，对应国家数据总量24903086.0
缺失占比为0.0
年份：1976，种类为Horse的fao总量为559522，对应国家数据总量5156830.0
年份：1976，种类为beef_cattle的fao总量为12800000，对应国家数据总量50784635.0
年份：1976，种类为Broilers的fao总量为549090000，对应国家数据总量168131980.0
年份：1976，种类为Pork - total的fao总量为11709000，对应国家数据总量38742102.0
年份：1976，种类为milk_cattle的fao总量为12852014，对应国家数据总量50784635.0
年份：1976，种类为Sheep_Goat的fao总量为25487160，对应国家数